# Week 7 · Notebook 1  RAG Policy Bot

**Chunk → embed → index → retrieve (vector → hybrid → rerank) → generate with citations, with recall@k measured separately from the answer.**

```
# Requirements: pip install chromadb sentence-transformers
```

```
# ⚠️ REQUIRES: internet for model download (CPU only); the GENERATION cell needs OPENAI_API_KEY / OPENROUTER_API_KEY or a local Ollama
```

Retrieval runs with a deterministic offline fallback if the model cannot download. Part of AI Engineering Lab · ZoroLogistics case study.

## Why RAG, and why measure retrieval separately

A foundation model knows what it was trained on, not ZoroLogistics' shipping policies. RAG grounds the answer in *your* corpus: retrieve relevant passages, then answer from them with citations. The rule that matters most: **measure retrieval (recall@k) separately from generation**  a low recall is an indexing/chunking problem, a low groundedness with high recall is a prompt problem, and you fix them in different places.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (assumes cwd == week-NN/)
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro").is_dir():
        sys.path.insert(0, str(_p)); break

from zoro import data
import numpy as np

docs = data.policy_docs()
print("loaded", len(docs), "policy documents:", [d["doc_id"] for d in docs])
print("\n--- POL-001 preview ---\n" + docs[0]["text"][:300])

## Section-aware chunking

The chunk is the unit of retrieval. We split on `## ` section headings rather than a fixed size, so each chunk is a coherent policy section (with its `doc_id` and section name carried as metadata  the citation is a pointer, not a paraphrase).

In [ ]:
def section_aware_chunks(doc):
    chunks = []
    current_heading, current_lines = "Overview", []
    for line in doc["text"].splitlines():
        if line.startswith("## "):
            if current_lines:
                chunks.append((current_heading, "\n".join(current_lines)))
            current_heading = line[3:].strip()
            current_lines = [line]
        else:
            current_lines.append(line)
    if current_lines:
        chunks.append((current_heading, "\n".join(current_lines)))
    return chunks

chunk_ids, doc_ids, sections, chunks = [], [], [], []
for d in docs:
    for i, (sec, text) in enumerate(section_aware_chunks(d)):
        chunk_ids.append(f"{d['doc_id']}::{i}")
        doc_ids.append(d["doc_id"])
        sections.append(sec)
        chunks.append(text)

print("total chunks:", len(chunks))
for i in range(6):
    print(f"  {chunk_ids[i]:14} | {doc_ids[i]} | {sections[i]:16} | {len(chunks[i])} chars")

## Embed + index

Embed every chunk with `all-MiniLM-L6-v2` (offline fallback included) and store vectors + text + metadata in Chroma. If Chroma is unavailable, we fall back to a brute-force NumPy index so the eval still runs.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    print("all-MiniLM-L6-v2 loaded")
except Exception as e:
    embedder = None
    print("⚠️ embedding model unavailable:", type(e).__name__, "-", e)
    print("   using deterministic hash-based fallback embedding.")

def embed(texts):
    if embedder is not None:
        return np.asarray(embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False), dtype=float)
    rng = np.random.default_rng(42)
    dim, vocab = 64, 4096
    proj = rng.normal(0.0, 1.0, size=(dim, vocab))
    out = []
    for t in texts:
        v = np.zeros(vocab)
        for w in t.lower().split():
            v[hash(w) % vocab] += 1.0
        p = proj @ v
        n = np.linalg.norm(p)
        out.append(p / n if n > 0 else np.zeros(dim))
    return np.stack(out)

def cosine(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return float(a @ b)

chunk_vecs = embed(chunks)
print("embedded", len(chunk_vecs), "chunks, dim =", chunk_vecs.shape[1])

CHROMA_OK = False
collection = None
try:
    import chromadb
    try:
        client = chromadb.EphemeralClient()
    except AttributeError:
        client = chromadb.Client()
    collection = client.get_or_create_collection("zoro_policy")
    collection.add(
        ids=chunk_ids,
        embeddings=[v.tolist() for v in chunk_vecs],
        documents=chunks,
        metadatas=[{"doc_id": d, "section": s} for d, s in zip(doc_ids, sections)],
    )
    CHROMA_OK = True
    print("indexed in Chroma:", collection.count(), "chunks")
except Exception as e:
    print("⚠️ Chroma unavailable:", type(e).__name__, "-", e)
    print("   using brute-force NumPy index for retrieval.")

## Retrieval + a golden eval set

Ten hand-written questions, each mapped to its relevant `doc_id`. `recall@k` = the fraction of questions whose relevant document is in the top-k retrieved chunks.

In [ ]:
QUERIES = [
    ("How late does a shipment have to be to get a 10% freight refund?", ["POL-002"]),
    ("Can I ship a 200 Wh lithium battery?", ["POL-003"]),
    ("How long do I have to file a damage claim?", ["POL-002"]),
    ("What is the fee to change the delivery address after pickup?", ["POL-001"]),
    ("What documents are needed for a cross-border shipment?", ["POL-004"]),
    ("What happens if customs holds my shipment for more than 5 days?", ["POL-004"]),
    ("Does severe weather extend delivery SLAs without penalty?", ["POL-001"]),
    ("What refund applies if a shipment is more than 7 days late?", ["POL-002"]),
    ("Are explosives allowed to be shipped?", ["POL-003"]),
    ("What must appear on the bill of lading for dangerous goods?", ["POL-003"]),
]

def vector_search(query, k=5):
    qv = embed([query])[0]
    if CHROMA_OK:
        res = collection.query(query_embeddings=[qv.tolist()], n_results=k,
                               include=["metadatas", "documents"])
        ids = res["ids"][0]; metas = res["metadatas"][0]; docs_ = res["documents"][0]
        return [{"chunk_id": ids[j], "doc_id": metas[j]["doc_id"], "section": metas[j]["section"],
                 "text": docs_[j]} for j in range(len(ids))]
    sims = [cosine(qv, v) for v in chunk_vecs]
    order = np.argsort(sims)[::-1][:k]
    return [{"chunk_id": chunk_ids[j], "doc_id": doc_ids[j], "section": sections[j],
             "text": chunks[j]} for j in order]

def recall_at_k(search_fn, k=5):
    hits = 0
    for q, rel in QUERIES:
        top = search_fn(q, k=k)
        if any(h["doc_id"] in rel for h in top):
            hits += 1
    return hits / len(QUERIES)

recall_vec = recall_at_k(vector_search, k=5)
print(f"vector-only recall@5: {recall_vec:.2f}")

## Hybrid search + rerank

Dense vectors catch *meaning* but miss *exact terms*; keyword (BM25-style) is the mirror image. We fuse the two with Reciprocal Rank Fusion, then rerank candidates by cross-score (cosine)  the "retrieve wide, rerank narrow" pattern.

In [ ]:
import re
_STOP = set("the a an and or of to for with in on by at from is are was were".split())

def keyword_score(query, text):
    terms = [w for w in re.findall(r"[a-z0-9]+", query.lower()) if w not in _STOP and len(w) > 2]
    tl = text.lower()
    return sum(1 for t in terms if t in tl)

def hybrid_search(query, k=5, candidate_pool=10):
    qv = embed([query])[0]
    # vector rank + keyword rank across the whole index
    vec_sims = np.array([cosine(qv, v) for v in chunk_vecs])
    kw_scores = np.array([keyword_score(query, t) for t in chunks])
    vec_rank = (-vec_sims).argsort().argsort()       # 0 = best
    kw_rank = (-kw_scores).argsort().argsort()
    rrf = 1.0 / (60.0 + vec_rank) + 1.0 / (60.0 + kw_rank)
    candidates = np.argsort(-rrf)[:candidate_pool]
    # rerank by cross-score (cosine) among the candidates
    reranked = sorted(candidates, key=lambda j: -vec_sims[j])[:k]
    return [{"chunk_id": chunk_ids[j], "doc_id": doc_ids[j], "section": sections[j],
             "text": chunks[j], "rrf": round(rrf[j], 4), "cross_score": round(vec_sims[j], 3)}
            for j in reranked]

recall_hybrid = recall_at_k(hybrid_search, k=5)
print(f"hybrid + rerank recall@5: {recall_hybrid:.2f}  (vector-only was {recall_vec:.2f})")
print("\nexample: top-3 for 'customs hold fees':")
for h in hybrid_search("What are the customs hold storage fees?", k=3):
    print(f"  [{h['chunk_id']} | {h['doc_id']} | {h['section']}] score={h['cross_score']}")

## Generation with citations

The final stage: feed the top chunks to a model and require a citation per claim. Works with a hosted key *or* a local Ollama model via the same OpenAI-compatible endpoint.

In [ ]:
import os, shutil

gen_mode = None
if os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENROUTER_API_KEY"):
    gen_mode = "openai"
elif shutil.which("ollama"):
    gen_mode = "ollama"

def build_context(hits):
    return "\n\n".join(f"[{h['chunk_id']} | {h['doc_id']} | {h['section']}]\n{h['text']}" for h in hits)

def generate(query):
    hits = hybrid_search(query, k=3)
    prompt = ("Answer only from the passages below. For each claim, cite the passage id in brackets. "
              "If the passages do not contain the answer, say 'not covered' and do not guess.\n\n"
              + build_context(hits) + "\n\nQuestion: " + query)
    if gen_mode == "openai":
        from openai import OpenAI
        if os.environ.get("OPENROUTER_API_KEY") and not os.environ.get("OPENAI_API_KEY"):
            cl = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")
            model = os.environ.get("OPENROUTER_MODEL", "deepseek/deepseek-chat")
        else:
            cl = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
            model = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
        resp = cl.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], temperature=0)
        return resp.choices[0].message.content
    if gen_mode == "ollama":
        from openai import OpenAI
        cl = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        model = os.environ.get("OLLAMA_MODEL", "llama3.2")
        resp = cl.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], temperature=0)
        return resp.choices[0].message.content
    return None

if gen_mode is None:
    print("⚠️ no API key and no Ollama, generation skipped. Retrieval eval above still ran.")
else:
    print("generation mode:", gen_mode)
    for q in ["What is the fee to change the delivery address after pickup?",
              "Can I ship a 200 Wh lithium battery?"]:
        ans = generate(q)
        print("\nQ:", q, "\nA:", ans)

In [ ]:
# Week 7 · Notebook 1 headline metric: recall@5 of the best retrieval config (hybrid + rerank).
print("WEEK7_NB1_RECALL_AT_5:", round(recall_hybrid, 3))